In [ ]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from transformers import GPT2TokenizerFast
from datasets import load_dataset
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time
import pandas as pd
import warnings

# Suppress HF warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

# ==========================================
# 1. ВВЕДЕДЕИЕ
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
D_MODEL = 512
MAX_SEQ_LEN = 128
BATCH_SIZE = 32
LR = 5e-4
TRAIN_STEPS = 1500
N_LAYERS = 8

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE = tokenizer.vocab_size

# ==========================================
# 2. ДЕТАДИ БДЕЙДИБГА
# ==========================================
class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * self.scale

class Qwen35RMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(hidden_size))
        self.variance_epsilon = eps

    def _norm(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.variance_epsilon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output = self._norm(x.float())
        output = output * (1.0 + self.weight.float())
        return output.type_as(x)

class PreNormBlock(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = RMSNorm(dim)
        self.ffn   = nn.Sequential(
            nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim)
        )

    def forward(self, x):
        mask = torch.triu(torch.ones(x.size(1), x.size(1), device=x.device), 1).bool()
        attn_out, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x), attn_mask=mask, need_weights=False)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x

class PostNormBlock(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.attn  = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm1 = Qwen35RMSNorm(dim)
        self.ffn   = nn.Sequential(
            nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim)
        )
        self.norm2 = Qwen35RMSNorm(dim)

    def forward(self, x):
        mask = torch.triu(torch.ones(x.size(1), x.size(1), device=x.device), 1).bool()
        attn_out, _ = self.attn(x, x, x, attn_mask=mask, need_weights=False)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x

class ResidualAdapter(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.proj  = nn.Linear(d_in, d_out, bias=False)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return self.gamma * self.proj(x)

class StandardTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers=N_LAYERS):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, MAX_SEQ_LEN, d_model) * 0.02)
        self.layers  = nn.ModuleList([PreNormBlock(d_model, 8) for _ in range(n_layers)])
        self.head    = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx) + self.pos_emb[:, :idx.size(1), :]
        for layer in self.layers:
            x = layer(x)
        return self.head(x)

class BowtieTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, d_small, n_layers=N_LAYERS):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, MAX_SEQ_LEN, d_model) * 0.02)
        self.layer_1    = PreNormBlock(d_model, 8)
        self.down_proj  = nn.Linear(d_model, d_small)
        self.entry_skip = ResidualAdapter(d_model, d_small)
        n_mid = n_layers - 2
        heads_small = 8 if d_small % 8 == 0 else 1
        self.middle_layers = nn.ModuleList([PreNormBlock(d_small, heads_small) for _ in range(n_mid)])
        self.up_proj      = nn.Linear(d_small, d_model)
        self.global_skip  = ResidualAdapter(d_model, d_model)
        self.exit_skip    = ResidualAdapter(d_small, d_model)
        self.layer_L = PreNormBlock(d_model, 8)
        self.head    = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x  = self.embed(idx) + self.pos_emb[:, :idx.size(1), :]
        h1 = self.layer_1(x)
        h_small = self.down_proj(h1) + self.entry_skip(h1)
        for layer in self.middle_layers:
            h_small = layer(h_small)
        h_big = self.up_proj(h_small) + self.global_skip(h1) + self.exit_skip(h_small)
        return self.head(self.layer_L(h_big))

def find_bowtie_d_small(target_params, vocab_size, d_model, n_layers=N_LAYERS):
    lo, hi = 32, d_model
    best = lo
    for _ in range(20):
        mid = ((lo + hi) // 16) * 8
        mid = max(8, mid)
        m = BowtieTransformer(vocab_size, d_model, mid, n_layers)
        p = sum(p.numel() for p in m.parameters())
        if p < target_params:
            lo = mid + 8
            best = mid
        else:
            hi = mid - 8
    return best

class HybridNormTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, d_small, n_layers=N_LAYERS):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, MAX_SEQ_LEN, d_model) * 0.02)
        self.layer_1    = PreNormBlock(d_model, 8)
        self.down_proj  = nn.Linear(d_model, d_small)
        self.entry_skip = ResidualAdapter(d_model, d_small)
        n_mid = n_layers - 2
        n_pre = max(1, round(n_mid * 0.25))
        n_post = n_mid - n_pre
        heads_small = 8 if d_small % 8 == 0 else 1
        self.middle_layers = nn.ModuleList([PreNormBlock(d_small, heads_small) for _ in range(n_pre)] + [PostNormBlock(d_small, heads_small) for _ in range(n_post)])
        self.up_proj      = nn.Linear(d_small, d_model)
        self.global_skip  = ResidualAdapter(d_model, d_model)
        self.exit_skip    = ResidualAdapter(d_small, d_model)
        self.layer_L = PostNormBlock(d_model, 8)
        self.head    = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x  = self.embed(idx) + self.pos_emb[:, :idx.size(1), :]
        h1 = self.layer_1(x)
        h_small = self.down_proj(h1) + self.entry_skip(h1)
        for layer in self.middle_layers:
            h_small = layer(h_small)
        h_big = self.up_proj(h_small) + self.global_skip(h1) + self.exit_skip(h_small)
        return self.head(self.layer_L(h_big))

class TinyStoriesIterable(IterableDataset):
    def __init__(self, dataset, tokenizer, max_len):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __iter__(self):
        for ex in self.dataset:
            ids = self.tokenizer.encode(ex["text"], truncation=True, max_length=self.max_len, padding="max_length")
            yield torch.tensor(ids)

print("У Вставив гараж дада...")
m_std = StandardTransformer(VOCAB_SIZE, D_MODEL)
target_params = sum(p.numel() for p in m_std.parameters())
D_SMALL = find_bowtie_d_small(target_params, VOCAB_SIZE, D_MODEL)

print("У Вставив дада...")
raw_dataset = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
iterable_ds = TinyStoriesIterable(raw_dataset, tokenizer, MAX_SEQ_LEN)
loader = DataLoader(iterable_ds, batch_size=BATCH_SIZE)

def train_model(model, name):
    print(f"\nУ Бучедие: {name}")
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == "cuda"))
    losses = []
    model.train()
    step = 0
    start = time.time()
    for batch in loader:
        x = batch.to(DEVICE)
        with torch.amp.autocast('cuda', enabled=(DEVICE == "cuda")):
            logits = model(x)
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, VOCAB_SIZE), x[:, 1:].reshape(-1), ignore_index=tokenizer.pad_token_id)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
        if step % 100 == 0:
            print(f"  Step {step}/{TRAIN_STEPS} | Loss: {loss.item():.4f}")
        step += 1
        if step >= TRAIN_STEPS: break
    return losses, sum(p.numel() for p in model.parameters())

m1 = StandardTransformer(VOCAB_SIZE, D_MODEL)
m2 = BowtieTransformer(VOCAB_SIZE, D_MODEL, D_SMALL)
m3 = HybridNormTransformer(VOCAB_SIZE, D_MODEL, D_SMALL)

loss_std, p1 = train_model(m1, "Standard")
loss_bow, p2 = train_model(m2, "Bowtie")
loss_hyb, p3 = train_model(m3, "Hybrid")

clear_output()
def smooth(x, w=50):
    return [sum(x[max(0,i-w):i+1]) / len(x[max(0,i-w):i+1]) for i in range(len(x))]

plt.figure(figsize=(12, 5))
plt.plot(smooth(loss_std), label=f"Standard ({p1/1e6:.1f}M)")
plt.plot(smooth(loss_bow), label=f"Bowtie ({p2/1e6:.1f}M)")
plt.plot(smooth(loss_hyb), label=f"Hybrid ({p3/1e6:.1f}M)")
plt.legend(); plt.grid(True); plt.show()

In [10]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def plot_advanced_metrics():
    # Check if we have at least some data to plot
    data_map = {
        'Standard': globals().get('loss_std', []),
        'Bowtie': globals().get('loss_bow', []),
        'Hybrid': globals().get('loss_hyb', [])
    }

    active_data = {k: v for k, v in data_map.items() if len(v) > 0}

    if not active_data:
        print("⏳ Обучение в процессе... Пока нет данных для отображения. Подождите хотя бы 100 шагов первой модели.")
        return

    plt.figure(figsize=(18, 10))

    # Subplot 1: Loss distribution
    plt.subplot(2, 2, 1)
    sns.boxplot(data=list(active_data.values()))
    plt.xticks(range(len(active_data)), list(active_data.keys()))
    plt.title('Loss Distribution (Current Progress)')
    plt.ylabel('Loss')

    # Subplot 2: Perplexity
    plt.subplot(2, 2, 2)
    ppl = [np.exp(np.mean(v[-50:])) for v in active_data.values()]
    plt.bar(list(active_data.keys()), ppl, color=['skyblue', 'salmon', 'lightgreen'][:len(active_data)])
    plt.title('Current Perplexity (Lower = Better)')
    plt.ylabel('PPL')

    # Subplot 3: Param Efficiency (if params known)
    plt.subplot(2, 2, 3)
    params = {'Standard': globals().get('p1', 0), 'Bowtie': globals().get('p2', 0), 'Hybrid': globals().get('p3', 0)}
    efficiency = [np.mean(v[-100:]) * (params[k]/1e6) for k, v in active_data.items() if params[k] > 0]
    if efficiency:
        plt.bar([k for k in active_data.keys() if params[k] > 0], efficiency, color=['blue', 'orange', 'green'])
        plt.title('Complexity Score (Loss * Params M)')

    # Subplot 4: Progress
    plt.subplot(2, 2, 4)
    for name, losses in active_data.items():
        plt.plot(losses, label=name, alpha=0.3)
        # Add smoothed line
        if len(losses) > 20:
            smooth = [np.mean(losses[max(0, i-20):i+1]) for i in range(len(losses))]
            plt.plot(smooth, linewidth=2)
    plt.title('Training Progress (Raw + Smooth)')
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_advanced_metrics()

⏳ Обучение в процессе... Пока нет данных для отображения. Подождите хотя бы 100 шагов первой модели.


In [2]:
import torch
try:
    from safetensors.torch import save_file, load_file
    HAS_SAFETENSORS = True
except ImportError:
    !pip install safetensors
    from safetensors.torch import save_file, load_file
    HAS_SAFETENSORS = True

def save_model_weights(model, filename_prefix="model"):
    # 1. Standard PyTorch format
    torch.save(model.state_dict(), f"{filename_prefix}.pth")
    print(f"Saved standard PyTorch weights to {filename_prefix}.pth")

    # 2. SafeTensors format (preferred for security/speed)
    save_file(model.state_dict(), f"{filename_prefix}.safetensors")
    print(f"Saved SafeTensors weights to {filename_prefix}.safetensors")

def load_model_weights(model, path, format="pt"):
    if format == "pt":
        model.load_state_dict(torch.load(path))
    elif format == "safetensors":
        state_dict = load_file(path)
        model.load_state_dict(state_dict)
    print(f"Model weights loaded from {path}")

# Example usage (commented out):
# save_model_weights(m3, "hybrid_transformer_final")
print("Weight management functions ready.")

Weight management functions ready.
